# Otimização de Rotas Médicas  - Entrega medicamentos

## Tech Challenge Fase 2

### Eduardo Toshio Yonamine - 9IADT - Grupo 37 - 14/07/2026

Sistema de otimização de rotas de entrega de médicamentos que usa algoritmos genéticos para resolver um VRP/TSP realista (múltiplos veículos, capacidade, autonomia, prioridades) e integra uma LLM para gerar relatórios operacionais e instruções para motoristas, operador e gerente.

----

# 1) Instalação de pacotes necessários

In [ ]:
!pip install numpy matplotlib groq python-dotenv --quiet
 
!pip install matplot
import sys
!{sys.executable} -m pip install plotly

import pandas as pd
from typing import List,Optional, Tuple, Dict
import numpy as np
import matplotlib.pyplot as plt
import random
import math
import copy
from dataclasses import dataclass
from rich import print_json
from rich.table import Table
from rich.console import Console

import json
import plotly.express as px
import plotly.graph_objects as go


----

## Configurações de parâmetros globais

**Gerais**

1.   Penalidades
2.   Quantidade, número, capacidade e autonomia dos veículos
3.   Probabilidades mutação e crossover
4.   Paciencia de espera de melhoria nas gerações
5.   Tamanho da população (rotas a gerar)
6.   Verbose quando indicado imprime cada melhoria

**Two Opt**

1.   Limite máximo de iterações para evitar loop infinito


**Heuristica Construtiva: Vizinho mais próximo(Nearest Neighbor) para TSP**

1.   Número de soluções
2.   Item da lista

In [ ]:
# ---------------------------------------------------------------------------
# Gerais
# ---------------------------------------------------------------------------
NUMERO_VEICULOS: 3
CAPACIDADE_VEICULO: 10.0
AUTONOMIA_VEICULO: 100.0
PROBABILIDADE_MUTACAO: float = 0.5
PROBABILIDADE_CROSSOVER: float = 1.0
VERBOSE = False

# ----------------------------------------------------------------------------
# Penalidades 
# ----------------------------------------------------------------------------
FATOR_PENALIDADE_PRIORIDADE = 5.0    # peso da penalidade de prioridade no custo
FATOR_PENALIDADE_CAPACIDADE = 17.0  # penalidade por unidade de carga acima da capacidade
FATOR_PENALIDADE_AUTONOMIA  = 1.0    # penalidade por pixel percorrido além da autonomia

# ----------------------------------------------------------------------------
# TWO OPT
PACIENCIA_ESPERA = 150 # gerações sem melhoria antes de repopular - Limite máximo de iterações para evitar loop infinito
# ----------------------------------------------------------------------------

# ----------------------------------------------------------------------------
# Heuristica Construtiva
# ----------------------------------------------------------------------------
TAMANHO_POPULACAO = 100
NUMERO_SOLUCOES = 1


--------

# 2) Hospital e lista de pontos de entrega

## 2.1) Classe PontoEntrega 

In [ ]:
@dataclass
class PontoEntrega:
    """Representa um ponto de entrega de medicamentos ou insumos."""
    nome: str
    coords: tuple          # (x, y) em pixels ou coordenadas normalizadas
    prioridade: int        # 1=Alta, 2=Média, 3=Baixa
    tempo_atendimento: int # minutos estimados no local
    peso: float = 1.0      # carga a entregar (unidades/kg); usada para restrição de capacidade
    is_origin: bool = False

    def __hash__(self):
        return hash((self.nome, self.coords))

    def __eq__(self, other):
        if isinstance(other, PontoEntrega):
            return self.coords == other.coords
        return False

# ---------------------------------------------------------------------------
# Dataset de exemplo: Hospital Base + 14 pontos de entrega
# ---------------------------------------------------------------------------
PONTOS_ENTREGA: List[PontoEntrega] = [
    PontoEntrega("Hospital Base",            coords=(512, 317), prioridade=0, tempo_atendimento=0,  peso=0.0, is_origin=True),
    PontoEntrega("UBS Vila Nova",            coords=(741,  72), prioridade=1, tempo_atendimento=15, peso=2.0),
    PontoEntrega("Clínica São Lucas",        coords=(552,  50), prioridade=2, tempo_atendimento=10, peso=1.5),
    PontoEntrega("Posto Saúde Centro",       coords=(772, 346), prioridade=1, tempo_atendimento=20, peso=3.0),
    PontoEntrega("Paciente - Rua das Flores",(637,  12),        prioridade=1, tempo_atendimento=10, peso=1.0),
    PontoEntrega("Farmácia Popular Norte",   coords=(589, 131), prioridade=3, tempo_atendimento=5,  peso=4.0),
    PontoEntrega("UBS Jardim América",       coords=(732, 165), prioridade=2, tempo_atendimento=15, peso=2.0),
    PontoEntrega("Paciente - Av. Brasil",    coords=(605,  15), prioridade=1, tempo_atendimento=10, peso=1.0),
    PontoEntrega("Clínica Bem Estar",        coords=(730,  38), prioridade=2, tempo_atendimento=10, peso=1.5),
    PontoEntrega("Posto Saúde Sul",          coords=(576, 216), prioridade=2, tempo_atendimento=20, peso=2.5),
    PontoEntrega("UBS Parque Verde",         coords=(589, 381), prioridade=3, tempo_atendimento=15, peso=3.0),
    PontoEntrega("Farmácia Central",         coords=(711, 387), prioridade=3, tempo_atendimento=5,  peso=4.0),
    PontoEntrega("Paciente - Rua do Lago",   coords=(563, 228), prioridade=1, tempo_atendimento=10, peso=1.0),
    PontoEntrega("Posto Saúde Leste",        coords=(494,  22), prioridade=2, tempo_atendimento=20, peso=2.0),
    PontoEntrega("UBS Vila Esperança",       coords=(787, 288), prioridade=3, tempo_atendimento=15, peso=3.0),
]

def get_hospital_base() -> PontoEntrega:
    """Retorna o ponto de origem (hospital base)."""
    return next(p for p in PONTOS_ENTREGA if p.is_origin)


def get_pontos_entrega_sem_origem() -> List[PontoEntrega]:
    """Retorna apenas os pontos de entrega, excluindo o hospital base."""
    return [p for p in PONTOS_ENTREGA if not p.is_origin]


# Mapeamento reutilizável de prioridade para rótulo textual
PRIORIDADE_LABEL = {1: "Alta", 2: "Média", 3: "Baixa"}



### 2.3) Demonstração

In [ ]:

print("Origem e retorno:", PONTOS_ENTREGA[0].nome)
print("Total de pontos de entrega:", len(PONTOS_ENTREGA))
i = 0
for p in PONTOS_ENTREGA:
    i = i + 1
    print(f"{i} - {p.nome} | coords={p.coords} | prioridade={p.prioridade} | peso={p.peso}")

---

# 3) Representação Genética

## (giant tour) + exemplo de cromossomo


In [ ]:
# Célula 3: Representação genética (giant tour)
# Roteiro: "Explico que o cromossomo é uma permutação de todos os pontos (sem depósito)."

# Exemplo de cromossomo (IDs ou nomes na ordem de visita)
pontos = get_pontos_entrega_sem_origem()
cromossomo_exemplo = [p.nome for p in pontos]  # ordem natural
# Para demonstrar, embaralhamos uma permutação exemplo
random.shuffle(cromossomo_exemplo)

print("Cromossomo (giant tour) exemplo:")
print(cromossomo_exemplo)


# 4) Operadores Genéticos 

Serão usados 4 operadores genéticos: 

***4.1) PMX (Partially Mapped Crossover***

***4.2) OX(Order Crossover)***

***4.3) Mutate (Mutação por troca adjacente )***

***4.4) Mutate Inversion (Mutação por inversão de segmento)***



<style>
  .titulo {
    font-size: 20px;
    font-weight: bold;
    padding: 10px;
    border-radius: 6px;
    margin-top: 25px;
    margin-bottom: 10px;
  }
  .crossover {
    background-color: #e3f2fd;
    border-left: 6px solid #2196f3;
  }
  .mutacao {
    background-color: #e8f5e9;
    border-left: 6px solid #4caf50;
  }
  .codigo {
    font-family: monospace;
    background-color: #f1f3f4;
    padding: 10px;
    border-radius: 6px;
    white-space: pre-wrap;
  }
</style>

---

## 🔵 **4.1) PMX — Partially Mapped Crossover**
<p>Preserva um segmento de um pai e usa mapeamento para preencher o restante sem duplicar pontos.</p>
<div class="codigo">
    
**Pai1:**

['UBS Vila Nova', 'Clínica São Lucas',<br>
'Farmácia Popular Norte', 'Posto Saúde Sul', 'Farmácia Central', 'UBS Vila Esperança']<br><br>
<b>Pai2:</b><br>
['Paciente - Rua do Lago', 'UBS Jardim América',<br>
'Farmácia Central', 'Posto Saúde Sul', 'Farmácia Popular Norte', 'Clínica Bem Estar']<br><br>
<b>Filho:</b><br>
['Paciente - Rua do Lago', 'UBS Jardim América',<br>
'Farmácia Popular Norte', 'Posto Saúde Sul', 'Farmácia Central', 'Clínica Bem Estar']
</div>

---


In [ ]:
def pmx_crossover(parent1: List[PontoEntrega], parent2: List[PontoEntrega]) -> List[PontoEntrega]:
    """
    Partially Mapped Crossover (PMX) para permutações.
    Garante filhos válidos sem duplicação de genes.
    """
    length = len(parent1)
    start, end = sorted(random.sample(range(length), 2))
    child = [None] * length

    # Copia segmento do primeiro pai
    child[start:end+1] = parent1[start:end+1]

    # Mapeamento
    for i in range(start, end+1):
        if parent2[i] not in child:
            pos = i
            while child[pos] is not None:
                pos = parent2.index(parent1[pos])
            child[pos] = parent2[i]

    # Preenche posições restantes
    for i in range(length):
        if child[i] is None:
            child[i] = parent2[i]

    return child


## 🔵 **4.2) OX — Order Crossover**

<p>Copia um trecho contínuo de um pai e completa com a ordem relativa do outro pai.</p>
<div class="codigo">
<b>Pai1:</b><br>
['UBS Vila Nova', 'Clínica São Lucas',<br>
'Farmácia Popular Norte', 'Posto Saúde Sul', 'Farmácia Central']<br><br>
<b>Pai2:</b><br>
['Posto Saúde Centro', 'Paciente - Av. Brasil',<br>
'Paciente - Rua das Flores', 'UBS Parque Verde', 'Posto Saúde Leste']<br><br>
<b>Filho:</b><br>
['Posto Saúde Centro', 'Paciente - Av. Brasil',<br>
'Farmácia Popular Norte', 'Posto Saúde Sul', 'UBS Parque Verde', 'Posto Saúde Leste']
</div>




In [ ]:
def order_crossover(
    parent1: List[PontoEntrega],
    parent2: List[PontoEntrega]
) -> List[PontoEntrega]:
    """
    Operador de crossover por ordem (OX — Order Crossover) para permutações.

    Preserva a estrutura de permutação válida, evitando duplicidade de cidades.

    Parâmetros:
    - parent1: rota do primeiro pai
    - parent2: rota do segundo pai

    Retorno:
    - rota filho resultante da recombinação
    """
    length = len(parent1)
    start_index = random.randint(0, length - 1)
    end_index = random.randint(start_index + 1, length)

    child = parent1[start_index:end_index]
    remaining_positions = [i for i in range(length) if i < start_index or i >= end_index]
    remaining_genes = [gene for gene in parent2 if gene not in child]

    for position, gene in zip(remaining_positions, remaining_genes):
        child.insert(position, gene)

    return child


---

## 🟢 **4.3)Mutação por Troca de Adjacentes**

<p>Troca dois pontos consecutivos da rota com certa probabilidade.</p>
<div class="codigo">
<b>Antes:</b><br>
['UBS Vila Nova', 'Clínica São Lucas', 'Farmácia Popular Norte', 'Posto Saúde Sul']<br><br>
<b>Depois:</b><br>
['UBS Vila Nova', 'Farmácia Popular Norte', 'Clínica São Lucas', 'Posto Saúde Sul']
</div>


In [ ]:
def mutate(
    rota: List[PontoEntrega],
    probabilidade_mutacao: float
) -> List[PontoEntrega]:
    """
    Operador de mutação: troca dois pontos adjacentes da rota com dada probabilidade.

    Preserva a validade da permutação.

    Parâmetros:
    - rota: sequência de pontos de entrega
    - probabilidade_mutacao: probabilidade de ocorrer mutação [0.0, 1.0]

    Retorno:
    - rota mutada (ou cópia da original se mutação não ocorrer)
    """
    rota_mutada = copy.deepcopy(rota)

    if random.random() < probabilidade_mutacao:
        if len(rota) < 2:
            return rota_mutada
        index = random.randint(0, len(rota) - 2)
        rota_mutada[index], rota_mutada[index + 1] = rota[index + 1], rota[index]

    return rota_mutada


---

## 🟢 **4.4)Mutação por Inversão de Segmento**

<p>Seleciona dois índices aleatórios e inverte o trecho entre eles.</p>
<div class="codigo">
<b>Antes:</b><br>
['UBS Vila Nova', 'Clínica São Lucas', 'Farmácia Popular Norte', 'Posto Saúde Sul', 'Farmácia Central']<br><br>
<b>Depois:</b><br>
['UBS Vila Nova', 'Clínica São Lucas', 'Farmácia Central', 'Posto Saúde Sul', 'Farmácia Popular Norte']
</div>

----
# 5) Funções de Fitness no Projeto VRP/TSP

O projeto utiliza duas funções de *fitness* para avaliar a qualidade das soluções geradas no problema de **roteamento de veículos (VRP)** e **caixeiro viajante (TSP)** aplicadas à entrega de medicamentos.

Essas funções medem o **custo total** de cada rota considerando:
- **Distância percorrida** (eficiência logística)
- **Penalidades** por violação de restrições operacionais:
  - Prioridade de pacientes
  - Capacidade de carga do veículo
  - Autonomia máxima de deslocamento

O objetivo do *fitness* é **minimizar o custo total**, favorecendo rotas curtas e viáveis que respeitem as restrições impostas.

As duas funções principais são:

***5.1. `calcular_custo_rota`*** → avalia uma rota individual.

***5.2. `calcular_custo_giant_tour_vrp`*** → avalia um conjunto de rotas (giant tour) para múltiplos veículos.

A seguir, cada função é detalhada com seus componentes e lógica de cálculo.


## 5.1) Função `calcular_custo_rota`

Avalia o custo de uma rota individual, incluindo o retorno ao hospital base.

**Componentes do custo:**
- **Distância total:** soma das distâncias entre todos os pontos da rota + retorno ao depósito.
- **Penalidade de prioridade:** pacientes críticos (prioridade 1 ou 2) atendidos mais tarde recebem penalidade proporcional à posição na rota.
- **Penalidade de capacidade:** aplicada quando o peso total excede a capacidade do veículo.
- **Penalidade de autonomia:** aplicada quando a distância total excede a autonomia do veículo.

**Fórmula geral:**
Custo total = Distância + Penalidade_prioridade + Penalidade_capacidade + Penalidade_autonomia


In [ ]:
# ---------------------------------------------------------------------------
# Função de custo (fitness)
# ---------------------------------------------------------------------------

def calcular_custo_rota(
    rota: List[PontoEntrega],
    hospital_base: PontoEntrega,
    fator_penalidade: float = FATOR_PENALIDADE_PRIORIDADE,
    capacidade_veiculo: Optional[float] = None,
    autonomia_veiculo: Optional[float] = None,
    fator_penalidade_capacidade: float = FATOR_PENALIDADE_CAPACIDADE,
    fator_penalidade_autonomia: float = FATOR_PENALIDADE_AUTONOMIA,
) -> float:
    """
    Calcula o custo total de uma rota incluindo distância e penalidades de restrições.

    Penalidades aplicadas:
    - Prioridade   : pacientes críticos atendidos tarde recebem penalidade proporcional
                     à (posição / prioridade) × fator_penalidade.
    - Capacidade   : excesso de carga além de ``capacidade_veiculo`` é penalizado de forma
                     proporcional ao excesso relativo (excesso/capacidade × fator).
    - Autonomia    : excesso de distância além de ``autonomia_veiculo`` é penalizado por
                     (excesso_pixels × fator_penalidade_autonomia).
    """
    rota_completa = [hospital_base] + list(rota) + [hospital_base]
    n = len(rota_completa)

    # Distância total do ciclo
    distancia_total = sum(
        calcular_distancia(rota_completa[i], rota_completa[i + 1])
        for i in range(n - 1)
    )

    # Penalidade por prioridade
    penalidade_prioridade = 0.0
    for posicao, ponto in enumerate(rota, start=1):
        if ponto.prioridade in (1, 2):
            penalidade_prioridade += (posicao / ponto.prioridade) * fator_penalidade

    # Penalidade proporcional por capacidade
    penalidade_capacidade = 0.0
    if capacidade_veiculo is not None:
        peso_total = sum(p.peso for p in rota)
        excesso = max(0.0, peso_total - capacidade_veiculo)
        if excesso > 0:
            excesso_relativo = excesso / capacidade_veiculo
            penalidade_capacidade = excesso_relativo * fator_penalidade_capacidade

    # Penalidade por autonomia
    penalidade_autonomia = 0.0
    if autonomia_veiculo is not None:
        excesso = max(0.0, distancia_total - autonomia_veiculo)
        penalidade_autonomia = excesso * fator_penalidade_autonomia

    return distancia_total + penalidade_prioridade + penalidade_capacidade + penalidade_autonomia



## 5.2 - Função - Calcular_custo_giant_tour_vrp

Avalia um *giant tour* (sequência única de pontos) e divide em sub-rotas para múltiplos veículos.

**Etapas principais:**
1. Percorre os pontos do *giant tour* acumulando peso e distância.
2. Se exceder capacidade ou autonomia, fecha a rota atual e inicia uma nova (se houver veículos disponíveis).
3. Calcula o custo de cada sub-rota usando `calcular_custo_rota`.
4. Soma todos os custos para obter o custo total do *giant tour*.

**Resultado final:**
Fitness total = Σ (custos das sub-rotas)


In [ ]:
def calcular_custo_giant_tour_vrp(
    giant_tour: List[PontoEntrega],
    hospital_base: PontoEntrega,
    n_veiculos: int,
    capacidade_veiculo: Optional[float] = None,
    autonomia_veiculo: Optional[float] = None,
    fator_penalidade: float = FATOR_PENALIDADE_PRIORIDADE,
    fator_penalidade_capacidade: float = FATOR_PENALIDADE_CAPACIDADE,
    fator_penalidade_autonomia: float = FATOR_PENALIDADE_AUTONOMIA,
) -> float:
    rotas: List[List[PontoEntrega]] = []
    rota_atual: List[PontoEntrega] = []
    peso_atual: float = 0.0
    dist_atual: float = 0.0
    ponto_anterior: PontoEntrega = hospital_base

    for ponto in giant_tour:
        dist_passo = calcular_distancia(ponto_anterior, ponto)
        dist_retorno = calcular_distancia(ponto, hospital_base)

        capacidade_violada = (
            capacidade_veiculo is not None
            and peso_atual + ponto.peso > capacidade_veiculo
        )
        autonomia_violada = (
            autonomia_veiculo is not None
            and dist_atual + dist_passo + dist_retorno > autonomia_veiculo
        )

        veiculos_disponiveis = len(rotas) < n_veiculos - 1
        if (capacidade_violada or autonomia_violada) and rota_atual and veiculos_disponiveis:
            rotas.append(rota_atual)
            rota_atual = []
            peso_atual = 0.0
            dist_atual = 0.0
            ponto_anterior = hospital_base
            dist_passo = calcular_distancia(hospital_base, ponto)

        rota_atual.append(ponto)
        peso_atual += ponto.peso
        dist_atual += dist_passo
        ponto_anterior = ponto

    if rota_atual:
        rotas.append(rota_atual)

    # Soma dos custos de cada sub-rota
    custo_total = 0.0
    for rota in rotas:
        custo_total += calcular_custo_rota(
            rota,
            hospital_base,            
            capacidade_veiculo,
            autonomia_veiculo,
            fator_penalidade_capacidade,
            fator_penalidade_autonomia,
        )
    return custo_total


***Fluxograma visual simplificado***
<div style="text-align:center">
    <img src="imagens/fitness.png" alt="Infográfico do fluxo de fitness" width="400">    
</div>


# 🚚 6) VRP Split e 🔧 Refinamento Local (Two‑Opt)

- O *giant tour* é uma sequência única de todos os pontos de entrega.
- O algoritmo **Split** divide esse tour em **sub‑rotas**, cada uma atribuída a um veículo.
- A divisão respeita **capacidade de carga** e **autonomia máxima** do veículo.
- Cada sub‑rota começa e termina no depósito (hospital).

**Exemplo:**
- Giant tour: `[Depósito, A, B, C, D, E, Depósito]`
- Split →  
  - Veículo 1: `[Depósito, A, B, Depósito]`  
  - Veículo 2: `[Depósito, C, D, E, Depósito]`

---

***Refinamento Local (Two‑Opt)***

- Objetivo: **reduzir a distância total** da rota trocando pares de arestas.
- Como funciona: remove duas conexões e reconecta os pontos em ordem diferente.
- Se a nova rota for mais curta, a mudança é aceita.
- Aplica‑se o Two‑Opt em cada sub‑rota obtida no Split.

**Antes (rota não refinada):**
Depósito → A → B → C → D → Depósito
Distância total = 120 km


**Depois (aplicando Two‑Opt, troca entre B e D):**
Depósito → A → D → C → B → Depósito
Distância total = 105 km


---

## Observações
- O **Split** garante rotas viáveis por veículo.
- O **Two‑Opt** melhora cada rota sem alterar os pontos visitados, apenas a ordem.
- Assim, obtemos soluções mais eficientes para o problema de rotas de entrega médica.



In [ ]:
# Célula: VRP Split final (dividir_rotas_vrp + calcular_custo_vrp + resumo_restricoes_vrp)
# Roteiro curto: "Decodificamos o giant tour em sub-rotas, avaliamos custo e checamos restrições."

def calcular_distancia(a: "PontoEntrega", b: "PontoEntrega") -> float:
    """Distância Euclidiana entre dois pontos (usa coords=(x,y))."""
    (ax, ay), (bx, by) = a.coords, b.coords
    return ((ax - bx) ** 2 + (ay - by) ** 2) ** 0.5

def calcular_custo_rota(
    rota: List["PontoEntrega"],
    hospital_base: "PontoEntrega",
    capacidade_veiculo: Optional[float] = None,
    autonomia_veiculo: Optional[float] = None,
    penalidade_cap: float = FATOR_PENALIDADE_CAPACIDADE,
    penalidade_auto: float = FATOR_PENALIDADE_AUTONOMIA,
) -> float:
    """
    Custo de uma rota: soma das distâncias (ida+volta) + penalidades por violação.
    Penalidades são altas para tornar soluções inviáveis menos atrativas ao GA.
    """
    if not rota:
        return 0.0
    rota_completa = [hospital_base] + rota + [hospital_base]
    distancia = sum(calcular_distancia(rota_completa[i], rota_completa[i + 1])
                    for i in range(len(rota_completa) - 1))
    peso_total = sum(p.peso for p in rota)
    custo = distancia
    if capacidade_veiculo is not None and peso_total > capacidade_veiculo:
        custo += penalidade_cap + 10.0 * (peso_total - capacidade_veiculo)
    if autonomia_veiculo is not None and distancia > autonomia_veiculo:
        custo += penalidade_auto + 10.0 * (distancia - autonomia_veiculo)
    return custo

def dividir_rotas_vrp(
    giant_tour: List["PontoEntrega"],
    hospital_base: "PontoEntrega",
    capacidade_veiculo: Optional[float] = None,
    autonomia_veiculo: Optional[float] = None,
    n_veiculos: Optional[int] = None,
    verbose: bool = False,
) -> Tuple[List[List["PontoEntrega"]], Dict]:
    """
    Divide um giant tour em sub-rotas válidas por veículo (greedy split).

    Retorna:
      - rotas: List[List[PontoEntrega]] (cada sub-rota não inclui o hospital)
      - meta: dict com metadados: n_veiculos_usados, n_rotas_infeasible, infeasible_indices
    """
    # Validações básicas
    if n_veiculos is not None and n_veiculos < 1:
        raise ValueError("n_veiculos deve ser >= 1 ou None.")
    # Filtra caso o giant_tour contenha o depósito por engano
    giant = [p for p in giant_tour if not getattr(p, "is_origin", False)]

    rotas: List[List["PontoEntrega"]] = []
    rota_atual: List["PontoEntrega"] = []
    peso_atual: float = 0.0
    dist_atual: float = 0.0
    ponto_anterior: "PontoEntrega" = hospital_base

    for idx, ponto in enumerate(giant, start=1):
        dist_passo = calcular_distancia(ponto_anterior, ponto)
        dist_retorno = calcular_distancia(ponto, hospital_base)

        capacidade_violada = (
            capacidade_veiculo is not None and (peso_atual + ponto.peso) > capacidade_veiculo
        )
        autonomia_violada = (
            autonomia_veiculo is not None and (dist_atual + dist_passo + dist_retorno) > autonomia_veiculo
        )

        veiculos_disponiveis = True
        if n_veiculos is not None:
            # permitimos abrir novas rotas enquanto já não tivermos fechado n_veiculos-1 rotas
            veiculos_disponiveis = len(rotas) < max(0, n_veiculos - 1)

        if verbose:
            print(f"[Passo {idx}] avaliando '{ponto.nome}': peso_atual={peso_atual:.1f}, dist_atual={dist_atual:.1f}")
            print(f"  dist_passo={dist_passo:.1f}, dist_retorno={dist_retorno:.1f}")
            if capacidade_violada:
                print("  -> violação de capacidade se adicionar este ponto.")
            if autonomia_violada:
                print("  -> violação de autonomia se adicionar este ponto.")

        # Se violaria e já há pontos na rota atual e ainda posso abrir novo veículo:
        if (capacidade_violada or autonomia_violada) and rota_atual and veiculos_disponiveis:
            if verbose:
                print("  >>> Abrindo nova rota (fechando rota atual).")
            rotas.append(rota_atual)
            rota_atual = []
            peso_atual = 0.0
            dist_atual = 0.0
            ponto_anterior = hospital_base
            # recalcula distância do depósito até este ponto
            dist_passo = calcular_distancia(hospital_base, ponto)
            if verbose:
                print(f"  >>> dist_passo recalculado desde depósito: {dist_passo:.1f}")

        rota_atual.append(ponto)
        peso_atual += ponto.peso
        dist_atual += dist_passo
        ponto_anterior = ponto

        if verbose:
            print(f"  -> ponto adicionado. rota atual tem {len(rota_atual)} pontos; peso_atual={peso_atual:.1f}, dist_atual={dist_atual:.1f}\n")

    if rota_atual:
        rotas.append(rota_atual)

    # Diagnóstico de infeasibilidades por rota (após construção)
    infeasible_indices = []
    for i, rota in enumerate(rotas):
        peso = sum(p.peso for p in rota)
        rota_completa = [hospital_base] + rota + [hospital_base]
        distancia = sum(calcular_distancia(rota_completa[j], rota_completa[j + 1]) for j in range(len(rota_completa) - 1))
        cap_ok = capacidade_veiculo is None or peso <= capacidade_veiculo
        auto_ok = autonomia_veiculo is None or distancia <= autonomia_veiculo
        if not (cap_ok and auto_ok):
            infeasible_indices.append(i)

    meta = {
        "n_veiculos_usados": len(rotas),
        "n_rotas_infeasible": len(infeasible_indices),
        "infeasible_indices": infeasible_indices,
    }

    if verbose:
        print(f"Split finalizado: veículos usados = {meta['n_veiculos_usados']}, rotas infeasible = {meta['n_rotas_infeasible']}")
    return rotas, meta

def calcular_custo_vrp(
    rotas_vrp: List[List["PontoEntrega"]],
    hospital_base: "PontoEntrega",
    capacidade_veiculo: Optional[float] = None,
    autonomia_veiculo: Optional[float] = None,
    penalidade_cap: float = FATOR_PENALIDADE_CAPACIDADE,
    penalidade_auto: float = 10,
) -> Tuple[float, List[float]]:
    """
    Calcula custo total e custos por veículo usando calcular_custo_rota.
    """
    custos = [
        calcular_custo_rota(
            rota, hospital_base,
            capacidade_veiculo=capacidade_veiculo,
            autonomia_veiculo=autonomia_veiculo,
            penalidade_cap=penalidade_cap,
            penalidade_auto=penalidade_auto,
        )
        for rota in rotas_vrp
    ]
    return sum(custos), custos

def resumo_restricoes_vrp(
    rotas_vrp: List[List[PontoEntrega]],
    hospital_base: PontoEntrega,
    capacidade_veiculo: Optional[float],
    autonomia_veiculo: Optional[float],
) -> List[dict]:
    """
    Gera um resumo das métricas de restrição por veículo.

    Retorno:
    - lista de dicionários com métricas de cada veículo
    """
    resumo = []
    for idx, rota in enumerate(rotas_vrp, start=1):
        peso = sum(p.peso for p in rota)
        rota_completa = [hospital_base] + rota + [hospital_base]
        distancia = sum(
            calcular_distancia(rota_completa[i], rota_completa[i + 1])
            for i in range(len(rota_completa) - 1)
        )
        capacidade_ok = capacidade_veiculo is None or peso <= capacidade_veiculo
        autonomia_ok = autonomia_veiculo is None or distancia <= autonomia_veiculo

        resumo.append({
            "veiculo": idx,
            "n_pontos": len(rota),
            "peso_total": round(peso, 2),
            "capacidade_veiculo": capacidade_veiculo,
            "capacidade_ok": capacidade_ok,
            "distancia_pixels": round(distancia, 2),
            "autonomia_veiculo": autonomia_veiculo,
            "autonomia_ok": autonomia_ok,
            "pontos": [p.nome for p in rota],
        })
    return resumo




***Demonstração da execução - Divisão das rotas***


In [ ]:

# -------------------------
# Demonstração rápida (execute durante o vídeo)
# -------------------------
# Observação: assume que get_hospital_base() e get_pontos_entrega_sem_origem() já existem
hospital = get_hospital_base()
pontos = get_pontos_entrega_sem_origem()

# Exemplo de cromossomo (giant tour) — ajuste a ordem para demonstrar diferentes splits
cromossomo_exemplo = [p.nome for p in pontos]  # ordem natural
# Para demo, embaralhe para mostrar comportamento (opcional)
import random
random.seed(42)
random.shuffle(cromossomo_exemplo)

# Mapeia nomes para objetos
mapa = {p.nome: p for p in pontos}
giant_tour = [mapa[n] for n in cromossomo_exemplo]

# Parâmetros de demo (reduza capacidade para forçar splits)
capacidade_veiculo = 8.0
autonomia_veiculo = 450.0
n_veiculos = 3

print("\n=== Executando dividir_rotas_vrp (versão final) ===\n")
rotas, meta = dividir_rotas_vrp(giant_tour, hospital, capacidade_veiculo, autonomia_veiculo, n_veiculos, verbose=True)

custo_total, custos_por_veiculo = calcular_custo_vrp(rotas, hospital, capacidade_veiculo, autonomia_veiculo)
resumo = resumo_restricoes_vrp(rotas, hospital, capacidade_veiculo, autonomia_veiculo)

print("\n=== Resultado do Split ===")
for i, rota in enumerate(rotas, start=1):
    nomes = [p.nome for p in rota]
    peso = sum(p.peso for p in rota)
    print(f"Veículo {i}: {nomes} | peso_total={peso:.1f}")

print(f"\nMeta: {meta}")
print(f"\nCusto total (com penalidades): {custo_total:.2f}")
print("Custos por veículo:", [round(c,2) for c in custos_por_veiculo])

print("\n=== Resumo de restrições por veículo ===")
for r in resumo:
    print(r)

# Observação final para o apresentador:
print("\nObservações:")
print("- O split é linear O(n) sobre o giant tour.")
print("- O último veículo pode ficar com rotas infeasible se n_veiculos limitar aberturas; meta['infeasible_indices'] indica quais.")
print("- Use 'verbose=True' para mostrar passo a passo durante a gravação.")

# 7)Sementes - Heurística Construtiva: Vizinho Mais Próximo (Nearest Neighbor)

O algoritmo **Nearest Neighbor (NN)** é uma heurística clássica para o Problema do Caixeiro Viajante (TSP), utilizada aqui como etapa inicial na construção de soluções para o VRP de entregas médicas.

### Passos do algoritmo:
1. Inicia no hospital base (ponto de origem).
2. A cada passo, visita o ponto de entrega ainda não visitado mais próximo do atual.
3. Ao esgotar todos os pontos, retorna ao hospital base (ciclo fechado).

### Complexidade:
- O algoritmo possui complexidade **O(n²)**, já que a cada passo é necessário buscar o ponto mais próximo entre os não visitados.

### Uso no pipeline:
- Gera soluções iniciais de qualidade superior às puramente aleatórias.
- Inicializa parte da população do Algoritmo Genético (GA), acelerando a convergência.
- Serve como **baseline** de comparação com a solução final (GA + Two Opt).

---
## Funções do método Vizinho Mais Próximo (NN)

Nesta seção temos três funções relacionadas ao algoritmo **Nearest Neighbor (NN)**:

1. **`nearest_neighbor`**  
   - Implementa o algoritmo clássico do vizinho mais próximo.  
   - Constrói **uma rota única**, partindo do hospital base e sempre escolhendo o ponto mais próximo ainda não visitado.  
   - Retorna apenas a sequência de pontos.

2. **`gerar_populacao_nearest_neighbor`**  
   - Usa o mesmo algoritmo NN, mas permite gerar **múltiplas rotas**.  
   - Faz isso variando o ponto de partida interno (após sair do hospital).  
   - É útil para criar uma **população inicial de soluções NN**, por exemplo, em algoritmos evolutivos como o GA.

3. **`avaliar_baseline_nn`**  
   - Função de **avaliação**.  
   - Executa o NN padrão (hospital como origem) e calcula o **custo total da rota**.  
   - Retorna `(rota_nn, custo_nn)` e serve como **baseline de comparação** contra métodos mais sofisticados (como GA + Two Opt).

---

In [ ]:

def nearest_neighbor(
    locais_entrega: List[PontoEntrega],
    hospital_base: PontoEntrega
) -> List[PontoEntrega]:
    """
    Constrói uma rota pelo método do Vizinho Mais Próximo.

    Parâmetros:
    - locais_entrega: pontos de entrega a visitar (sem o hospital base)
    - hospital_base: ponto de partida e retorno

    Retorno:
    - rota ordenada de pontos de entrega (sem o hospital base nos extremos)
    """
    nao_visitados = list(locais_entrega)
    rota: List[PontoEntrega] = []
    ponto_atual: PontoEntrega = hospital_base

    while nao_visitados:
        mais_proximo = min(nao_visitados,
                           key=lambda p: calcular_distancia(ponto_atual, p))
        rota.append(mais_proximo)
        nao_visitados.remove(mais_proximo)
        ponto_atual = mais_proximo

    return rota


def gerar_populacao_nearest_neighbor(
    locais_entrega: List[PontoEntrega],
    hospital_base: PontoEntrega,
    n_solucoes: int = 1
) -> List[List[PontoEntrega]]:
    """
    Gera múltiplas soluções via Nearest Neighbor com pontos de partida variados.

    Para n_solucoes > 1, utiliza pontos de entrega aleatórios como ponto de
    partida interno (após sair do hospital base), gerando variações do NN.

    Parâmetros:
    - locais_entrega: pontos de entrega (sem o hospital base)
    - hospital_base: ponto de origem e retorno
    - n_solucoes: número de soluções NN a gerar

    Retorno:
    - lista de rotas geradas pelo método NN
    """
    import random

    solucoes: List[List[PontoEntrega]] = []

    # Primeira solução: NN padrão iniciando do hospital base
    solucoes.append(nearest_neighbor(locais_entrega, hospital_base))

    # Demais soluções: NN iniciando de pontos de partida internos aleatórios
    for _ in range(n_solucoes - 1):
        ponto_partida = random.choice(locais_entrega)
        nao_visitados = [p for p in locais_entrega if p != ponto_partida]
        rota = [ponto_partida]
        ponto_atual = ponto_partida

        while nao_visitados:
            mais_proximo = min(nao_visitados,
                               key=lambda p: calcular_distancia(ponto_atual, p))
            rota.append(mais_proximo)
            nao_visitados.remove(mais_proximo)
            ponto_atual = mais_proximo

        solucoes.append(rota)

    return solucoes


def avaliar_baseline_nn(
    locais_entrega: List[PontoEntrega],
    hospital_base: PontoEntrega
) -> Tuple[List[PontoEntrega], float]:
    """
    Retorna a rota NN padrão e seu custo, para uso como baseline de comparação.

    Parâmetros:
    - locais_entrega: pontos de entrega (sem o hospital base)
    - hospital_base: ponto de origem e retorno

    Retorno:
    - (rota_nn, custo_nn)
    """
    rota_nn = nearest_neighbor(locais_entrega, hospital_base)
    custo_nn = calcular_custo_rota(rota_nn, hospital_base)
    return rota_nn, custo_nn


### Exemplo - executando o calculo e gerando gráfico

* Usa o hospital base como origem.

* Calcula a rota NN padrão e imprime o custo.

* Mostra a sequência de visitas (com coordenadas e prioridade).

* Gera 5 soluções NN diferentes para inicializar a população do GA, variando o ponto de partida interno.

* Imprime cada rota como uma sequência de nomes, para fácil visualização

In [ ]:
# ---------------------------------------------------------------------------
# Exemplo de uso da heurística Nearest Neighbor com o dataset de entregas
# ---------------------------------------------------------------------------

# Obter hospital base e pontos de entrega
hospital_base = get_hospital_base()
locais_entrega = get_pontos_entrega_sem_origem()

# Gerar rota baseline via NN
rota_nn, custo_nn = avaliar_baseline_nn(locais_entrega, hospital_base)

print("Rota NN (baseline):")
for p in rota_nn:
    print(f"- {p.nome} ({p.coords}) | prioridade={PRIORIDADE_LABEL.get(p.prioridade, 'Origem')}")

print(f"\nCusto total da rota NN: {custo_nn:.2f}")

# Gerar múltiplas soluções NN para inicializar população do GA
populacao_inicial = gerar_populacao_nearest_neighbor(locais_entrega, hospital_base, n_solucoes=5)

print("\nExemplos de rotas NN para população inicial:")
for i, rota in enumerate(populacao_inicial, start=1):
    nomes = " -> ".join([p.nome for p in rota])
    print(f"Solução {i}: {nomes}")

# ---------------------------------------------------------------------------
# Visualização gráfica da rota NN
# ---------------------------------------------------------------------------
# Obter hospital base e pontos de entrega
hospital_base = get_hospital_base()
locais_entrega = get_pontos_entrega_sem_origem()

# Calcular rota NN
rota_nn, custo_nn = avaliar_baseline_nn(locais_entrega, hospital_base)

# Preparar coordenadas
x_coords = [hospital_base.coords[0]] + [p.coords[0] for p in rota_nn] + [hospital_base.coords[0]]
y_coords = [hospital_base.coords[1]] + [p.coords[1] for p in rota_nn] + [hospital_base.coords[1]]

# Plotar pontos
plt.figure(figsize=(8,6))
plt.scatter([p.coords[0] for p in locais_entrega], 
            [p.coords[1] for p in locais_entrega], 
            c="blue", label="Pontos de Entrega")

plt.scatter(hospital_base.coords[0], hospital_base.coords[1], 
            c="red", marker="s", s=100, label="Hospital Base")

# Plotar rota NN
plt.plot(x_coords, y_coords, c="green", linestyle="--", linewidth=2, label="Rota NN")

# Anotar nomes dos pontos
for p in locais_entrega:
    plt.text(p.coords[0]+5, p.coords[1]+5, p.nome, fontsize=8)

plt.text(hospital_base.coords[0]+5, hospital_base.coords[1]+5, hospital_base.nome, fontsize=9, fontweight="bold")

plt.title(f"Rota NN - Custo total: {custo_nn:.2f}")
plt.xlabel("X (coordenadas)")
plt.ylabel("Y (coordenadas)")
plt.legend()
plt.grid(True)
plt.show()    


# 8) Algoritimo Genético

In [ ]:
def executar_algoritmo_genetico(
    locais_entrega: List[PontoEntrega],
    hospital_base: PontoEntrega,
    populacao_inicial: List[List[PontoEntrega]],
    probabilidade_mutacao: float = 0.4,
    probabilidade_crossover: float = 1.0,
    tamanho_elite: int = 10,
    paciencia: int = 150,
    tolerancia: float = 1e-4,
    n_veiculos: Optional[int] = None,
    capacidade_veiculo: Optional[float] = None,
    autonomia_veiculo: Optional[float] = None,
    fator_penalidade: float = FATOR_PENALIDADE_PRIORIDADE,
    fator_penalidade_capacidade: float = FATOR_PENALIDADE_CAPACIDADE,
    fator_penalidade_autonomia: float = FATOR_PENALIDADE_AUTONOMIA,
    verbose: bool = True,
    limite_tempo: Optional[int] = 120  # tempo máximo em segundos
) -> Tuple[List[PontoEntrega], float, List[float], List[float]]:
    import numpy as np
    import time

    inicio = time.time()

    # Fitness adaptado para VRP
    _vrp_mode = n_veiculos is not None and (
        capacidade_veiculo is not None or autonomia_veiculo is not None
    )

    def _custo(rota):
        if _vrp_mode:
            return calcular_custo_giant_tour_vrp(
                rota, hospital_base,
                n_veiculos=n_veiculos,
                capacidade_veiculo=capacidade_veiculo,
                autonomia_veiculo=autonomia_veiculo,
                fator_penalidade=fator_penalidade,
                fator_penalidade_capacidade=fator_penalidade_capacidade,
                fator_penalidade_autonomia=fator_penalidade_autonomia,
            )
        return calcular_custo_rota(
            rota, 
            hospital_base,            
            capacidade_veiculo,
            autonomia_veiculo,            
            fator_penalidade_capacidade,
            fator_penalidade_autonomia,
        )

    populacao_rotas = list(populacao_inicial)
    historico_custos, historico_media = [], []

    custos_iniciais = [_custo(rota) for rota in populacao_rotas]
    idx_melhor = custos_iniciais.index(min(custos_iniciais))
    melhor_custo_global = custos_iniciais[idx_melhor]
    melhor_rota_global = populacao_rotas[idx_melhor]
    geracoes_sem_melhora = 0
    geracao = 0

    while True:  # loop infinito até critério híbrido parar
        geracao += 1
        custos = [_custo(rota) for rota in populacao_rotas]
        populacao_rotas, custos = ordenar_populacao(populacao_rotas, custos)

        melhor_custo, melhor_rota = custos[0], populacao_rotas[0]
        historico_custos.append(melhor_custo)
        historico_media.append(float(np.mean(custos)))

        if melhor_custo_global - melhor_custo > tolerancia:
            melhoria = melhor_custo_global - melhor_custo
            melhor_custo_global, melhor_rota_global = melhor_custo, melhor_rota
            geracoes_sem_melhora = 0
            if verbose:
                print(f"Geração {geracao:>4}: melhor custo = {melhor_custo:.2f} | melhoria detectada: {melhoria:.2f}")
        else:
            geracoes_sem_melhora += 1
            if verbose:
                print(f"Geração {geracao:>4}: melhor custo = {melhor_custo:.2f} (sem melhoria: {geracoes_sem_melhora}/{paciencia})")

        # Critérios híbridos de parada
        if geracoes_sem_melhora >= paciencia:
            print(f"\n⏹  Parada antecipada na geração {geracao}: sem melhoria por {paciencia} gerações consecutivas.")
            break
        if np.std(custos) < 1e-3:
            print(f"\n⏹  Parada antecipada na geração {geracao}: população convergiu (baixa diversidade).")
            break
        if limite_tempo is not None and (time.time() - inicio > limite_tempo):
            print(f"\n⏹  Parada antecipada na geração {geracao}: limite de tempo atingido ({limite_tempo}s).")
            break

        # Seleção com elitismo
        pesos_selecao = 1.0 / np.array(custos)
        nova_populacao = populacao_rotas[:tamanho_elite]

        while len(nova_populacao) < len(populacao_rotas):
            parent1, parent2 = random.choices(populacao_rotas, weights=pesos_selecao, k=2)
            if parent1 == parent2:
                parent2 = random.choices(populacao_rotas, weights=pesos_selecao, k=1)[0]

            # Crossover híbrido
            if random.random() < probabilidade_crossover:
                filho = order_crossover(parent1, parent2) if random.random() < 0.5 else pmx_crossover(parent1, parent2)
            else:
                filho = list(parent1)

            # Mutação adaptativa
            taxa_mutacao_atual = probabilidade_mutacao
            if geracoes_sem_melhora > 20:
                taxa_mutacao_atual = min(1.0, probabilidade_mutacao * 1.5)
                if verbose:
                    print(f"⚡ Mutação adaptativa ativada na geração {geracao}: taxa = {taxa_mutacao_atual:.2f}")

            filho = mutate(filho, taxa_mutacao_atual) if random.random() < 0.5 else mutate_segment_inversion(filho, taxa_mutacao_atual)
            nova_populacao.append(filho)

        # Reinicialização parcial
        if geracoes_sem_melhora > paciencia // 2:
            if verbose:
                print(f"🔄 Reinicialização parcial na geração {geracao}: adicionando novos indivíduos.")
            for _ in range(len(populacao_rotas) // 4):
                nova_populacao.append(random.sample(locais_entrega, len(locais_entrega)))

        populacao_rotas = nova_populacao

    return melhor_rota_global, melhor_custo_global, historico_custos, historico_media


## Executar GA

### Funções - Gerar e ordenar população aleatória

In [ ]:
def gerar_populacao_aleatoria(
    locais_entrega: List[PontoEntrega],
    tamanho_populacao: int
) -> List[List[PontoEntrega]]:
    """
    Gera uma população de rotas aleatórias por permutação dos locais de entrega.

    O hospital base NÃO é incluído nas rotas — ele é tratado como origem fixa.

    Parâmetros:
    - locais_entrega: pontos de entrega (sem o hospital base)
    - tamanho_populacao: número de rotas a gerar

    Retorno:
    - lista de rotas (cada rota é uma permutação dos locais_entrega)
    """
    return [random.sample(locais_entrega, len(locais_entrega))
            for _ in range(tamanho_populacao)]

# ---------------------------------------------------------------------------
# Ordenação da população
# ---------------------------------------------------------------------------

def ordenar_populacao(
    populacao_rotas: List[List[PontoEntrega]],
    custos: List[float]
) -> Tuple[List[List[PontoEntrega]], List[float]]:
    """
    Ordena a população de rotas do menor custo (melhor) para o maior (pior).

    Parâmetros:
    - populacao_rotas: lista de rotas
    - custos: lista de custos correspondentes a cada rota

    Retorno:
    - tupla (populacao_ordenada, custos_ordenados)
    """
    combinado = sorted(zip(populacao_rotas, custos), key=lambda x: x[1])
    populacao_ordenada, custos_ordenados = zip(*combinado)
    return list(populacao_ordenada), list(custos_ordenados)

In [ ]:
hospital_base_param = get_hospital_base()
locais_entrega_param = get_pontos_entrega_sem_origem()

PROPORCAO_NN = 0.15  # 15% da população inicial gerada por Nearest Neighbor

n_nn = max(1, int(TAMANHO_POPULACAO * PROPORCAO_NN))
n_aleatorio = TAMANHO_POPULACAO - n_nn

pop_nn = gerar_populacao_nearest_neighbor(locais_entrega_param, hospital_base_param, n_nn)
pop_aleatoria = gerar_populacao_aleatoria(locais_entrega_param, n_aleatorio)
populacao_inicial = pop_nn + pop_aleatoria

## Demonstração de experimentos com parametrizações do Algoritmo Genético

Vamos explorar como diferentes configurações de parâmetros afetam o desempenho do algoritmo genético. Abaixo, executaremos o GA com três conjuntos distintos de parâmetros.

### Configuração 1: Cenário Base (Gerações e Veículos Padrão)

In [ ]:
# Configuração 1: Cenário Base
print("\n--- Executando GA com Configuração 1 ---")
melhor_rota_cfg1, custo_cfg1, histcutomedio1, _ = executar_algoritmo_genetico(
    locais_entrega = locais_entrega_param,
    hospital_base = hospital_base_param,
    populacao_inicial = populacao_inicial,            
    probabilidade_mutacao= 0.7,
    probabilidade_crossover= 2.0,
    tamanho_elite = 10,
    paciencia = 150,
    tolerancia = 1e-6,
    n_veiculos= 3,
    capacidade_veiculo=6.0,
    autonomia_veiculo=20.0,
    fator_penalidade = FATOR_PENALIDADE_PRIORIDADE,
    fator_penalidade_capacidade = FATOR_PENALIDADE_CAPACIDADE,
    fator_penalidade_autonomia = FATOR_PENALIDADE_AUTONOMIA,
    verbose=False # Desabilitar verbose para melhor visualização na demonstração
)
print("Configuração 1 - Melhor rota encontrada:", melhor_rota_cfg1)
print("Configuração 1 - Custo total:", custo_cfg1)

### Configuração 2: Mais Veículos, Restrições Mais Apertadas

In [ ]:
# Configuração 2: Mais Veículos, Restrições Mais Apertadas

FATOR_PENALIDADE_PRIORIDADE_2 = 1.0
FATOR_PENALIDADE_CAPACIDADE_2 = 10.0
FATOR_PENALIDADE_AUTONOMIA_3 = 3.0

print("\n--- Executando GA com Configuração 2 ---")
melhor_rota_cfg2, custo_cfg2, histcutomedio2, _ = executar_algoritmo_genetico(
    locais_entrega_param,
    hospital_base_param,
    populacao_inicial=populacao_inicial,    
    probabilidade_mutacao=0.6,
    probabilidade_crossover=1.0,
    tamanho_elite = 20,
    paciencia = 300,
    tolerancia = 1e-4,
    n_veiculos= 6,  # Mais veículos
    capacidade_veiculo=4.0, # Capacidade mais apertada
    autonomia_veiculo=15.0, # Autonomia mais apertada
    fator_penalidade = FATOR_PENALIDADE_PRIORIDADE,
    fator_penalidade_capacidade = FATOR_PENALIDADE_CAPACIDADE,
    fator_penalidade_autonomia = FATOR_PENALIDADE_AUTONOMIA,
    verbose=False
)
print("Configuração 2 - Melhor rota encontrada:", melhor_rota_cfg2)
print("Configuração 2 - Custo total:", custo_cfg2)

### Configuração 3: Cenário TSP (Um Veículo, Sem Restrições de Capacidade/Autonomia)

In [ ]:
# Configuração 3: Cenário TSP (Um Veículo, Sem Restrições)
print("\n--- Executando GA com Configuração 3 ---")
melhor_rota_cfg3, custo_cfg3, histcutomedio3, _ = executar_algoritmo_genetico(
    locais_entrega_param,
    hospital_base_param,
    populacao_inicial=populacao_inicial,    
    probabilidade_mutacao=0.4,
    probabilidade_crossover=1.0,
    tamanho_elite = 30,
    paciencia = 150,
    tolerancia = 1e-6,
    n_veiculos=3,  # Apenas um veículo (TSP)
    capacidade_veiculo=None, # Sem restrição de capacidade
    autonomia_veiculo=None,  # Sem restrição de autonomia
    verbose=False
)
print("Configuração 3 - Melhor rota encontrada:", melhor_rota_cfg3)
print("Configuração 3 - Custo total:", custo_cfg3)

## Plot com históricos de custo

In [ ]:
def plot_comparacao_execucoes(historicos, labels, title="Comparação de execuções"):
    plt.figure(figsize=(10,6))
    for hist, label in zip(historicos, labels):
        gens = list(range(1, len(hist)+1))
        plt.plot(gens, hist, label=label)
    plt.xlabel("Geração")
    plt.ylabel("Custo da rota")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Chamando a função
plot_comparacao_execucoes(
    historicos=[histcutomedio1, histcutomedio2, histcutomedio3],
    labels=["Execução 1 - Configuração 1", "Execução 2 - Configuração 2", "Execução 3 - Configuração 3"]
)

# 9) Refinamento: Two Opt Inversion

In [ ]:
def two_opt_inversion(
    rota: List[PontoEntrega],
    hospital_base: PontoEntrega,
    max_iteracoes: int = 1000,
    verbose: bool = False
) -> Tuple[List[PontoEntrega], float]:
    """
    Aplica o Two Opt Inversion sobre uma rota para refinamento local.

    Para cada par de arestas (i, k), testa se inverter o segmento [i+1 .. k]
    reduz o custo total. Aceita imediatamente a primeira melhoria encontrada
    (estratégia first improvement) e reinicia a busca.

    Parâmetros:
    - rota: sequência de pontos de entrega (sem o hospital base nos extremos)
    - hospital_base: ponto de origem e retorno (usado no cálculo do custo)
    - max_iteracoes: limite máximo de iterações para evitar loop infinito
    - verbose: se True, imprime cada melhoria encontrada

    Retorno:
    - (rota_otimizada, custo_otimizado)
    """
    melhor_rota = list(rota)
    melhor_custo = calcular_custo_rota(melhor_rota, hospital_base)
    n = len(melhor_rota)
    iteracao = 0
    melhoria_encontrada = True

    while melhoria_encontrada and iteracao < max_iteracoes:
        melhoria_encontrada = False
        iteracao += 1

        for i in range(n - 1):
            for k in range(i + 1, n):
                # Gera nova rota invertendo o segmento entre i+1 e k
                nova_rota = melhor_rota[:i + 1] + \
                            melhor_rota[i + 1:k + 1][::-1] + \
                            melhor_rota[k + 1:]

                novo_custo = calcular_custo_rota(nova_rota, hospital_base)

                if novo_custo < melhor_custo:
                    melhor_rota = nova_rota
                    melhor_custo = novo_custo
                    melhoria_encontrada = True

                    if verbose:
                        print(f"  Two Opt iter {iteracao}: inversão [{i+1}:{k+1}] "
                              f"→ custo = {melhor_custo:.2f}")
                    break  # first improvement: reinicia

            if melhoria_encontrada:
                break

    return melhor_rota, melhor_custo

In [ ]:
melhor_rota_refinado, melhor_custo_refinado = two_opt_inversion( melhor_rota_cfg1, hospital_base_param, 1000, True)

print(f"\n✅ Custo final (GA + Two Opt): {custo_cfg1:.2f}")
if custo_cfg1 > 0:
    melhoria_two_opt = custo_cfg1 - melhor_custo_refinado
    print(f"📉 Melhoria Two Opt: {melhoria_two_opt:.2f} ({100-(melhoria_two_opt/custo_cfg1)*100:.1f}%)")

# 10) VRP Split: particionamento em rotas por veículo

In [ ]:
CAPACIDADE_VEICULO = 6.0
AUTONOMIA_VEICULO = 20.0
NUMERO_VEICULOS = 3


rotas_vrp, meta = dividir_rotas_vrp(
    giant_tour = melhor_rota_refinado,
    hospital_base = hospital_base_param,
    capacidade_veiculo = CAPACIDADE_VEICULO,
    autonomia_veiculo= AUTONOMIA_VEICULO,
    n_veiculos= NUMERO_VEICULOS,
)

resumo_vrp = resumo_restricoes_vrp(
    rotas_vrp, hospital_base_param , CAPACIDADE_VEICULO, AUTONOMIA_VEICULO
)

emoji_prioridade = {1: "🔴", 2: "🟡", 3: "🟢"}
print(f"\n🚐 Solução VRP: {len(rotas_vrp)} veículo(s) necessário(s)\n")
for v in resumo_vrp:
    cap_ok = "✅" if v["capacidade_ok"] else "❌"
    aut_ok = "✅" if v["autonomia_ok"] else "❌"
    print(f"  Veículo {v['veiculo']}:")
    print(f"    Pontos       : {v['n_pontos']}")
    print(f"    Carga total  : {v['peso_total']} / {v['capacidade_veiculo']} un.  {cap_ok}")
    print(f"    Distância    : {v['distancia_pixels']:.0f} / {v['autonomia_veiculo']} px  {aut_ok}")
    rota_v = rotas_vrp[v["veiculo"] - 1]
    rota_str = hospital_base.nome
    for ponto in rota_v:
        prio = emoji_prioridade.get(ponto.prioridade, "⚪")
        rota_str += f" → {prio} {ponto.nome}"
    rota_str += f" → {hospital_base.nome}"
    print(f"    Rota         : {rota_str}\n")

# 11) Relatório com Groq

## Configuração da chave da API do Groq para uso no LLM

No Colab, use input() para digitar a chave manualmente

In [ ]:
from getpass import getpass

api_key = getpass("Digite sua chave Groq: ")
print("Chave recebida com sucesso!")  # só para confirmar, não imprime a chave


## Geração do relatório com API Groq

In [ ]:
# Velocidade média estimada para cálculo de tempo de deslocamento (km/h → pixels/min)
# Em ambiente de demonstração, tratamos 1 unidade de pixel ≈ 0.1 km
VELOCIDADE_MEDIA_PIXELS_POR_MINUTO = 5.0

def gerar_relatorio_rota(
    rota_otimizada: List[PontoEntrega],
    hospital_base: PontoEntrega,
    custo_otimizado: float,
    historico_custos: List[float],
    rota_baseline_nn: Optional[List[PontoEntrega]] = None,
    custo_baseline_nn: Optional[float] = None,
    rotas_vrp: Optional[List[List[PontoEntrega]]] = None,
    resumo_vrp: Optional[List[dict]] = None,
) -> Dict[str, Any]:
    """
    Gera um relatório estruturado com os dados da rota otimizada.
    """

    rota_completa = [hospital_base] + list(rota_otimizada) + [hospital_base]
    n = len(rota_completa)

    # Distância pura (sem penalidade de prioridade)
    distancia_total = sum(
        calcular_distancia(rota_completa[i], rota_completa[i + 1])
        for i in range(n - 1)
    )
    tempo_total_estimado = distancia_total / VELOCIDADE_MEDIA_PIXELS_POR_MINUTO + sum(
        p.tempo_atendimento for p in rota_otimizada
    )

    # Sequência detalhada de atendimentos
    sequencia_atendimentos = []
    tempo_acumulado = 0.0
    ponto_anterior = hospital_base
    for posicao, ponto in enumerate(rota_otimizada, start=1):
        deslocamento = calcular_distancia(ponto_anterior, ponto)
        tempo_desloc = deslocamento / VELOCIDADE_MEDIA_PIXELS_POR_MINUTO
        tempo_acumulado += tempo_desloc + ponto.tempo_atendimento
        sequencia_atendimentos.append({
            "posicao": posicao,
            "nome": ponto.nome,
            "coords": ponto.coords,
            "prioridade": ponto.prioridade,
            "prioridade_label": PRIORIDADE_LABEL.get(ponto.prioridade, "-"),
            "tempo_atendimento_min": ponto.tempo_atendimento,
            "tempo_deslocamento_min": round(tempo_desloc, 1),
            "tempo_acumulado_min": round(tempo_acumulado, 1),
        })
        ponto_anterior = ponto

    # Métricas de comparação com baseline NN
    economia_percentual = None
    if custo_baseline_nn and custo_baseline_nn > 0:
        economia_percentual = round(
            (custo_baseline_nn - custo_otimizado) / custo_baseline_nn * 100, 2
        )

    # Contagem de melhorias vs estagnações
    n_geracoes_total = len(historico_custos)
    n_geracoes_com_melhoria = sum(
        1 for i in range(1, n_geracoes_total)
        if historico_custos[i] < historico_custos[i-1]
    )
    n_geracoes_sem_melhoria = n_geracoes_total - n_geracoes_com_melhoria

    relatorio = {
        "resumo": {
            "total_pontos_entrega": len(rota_otimizada),
            "distancia_total_pixels": round(distancia_total, 2),
            "tempo_total_estimado_min": round(tempo_total_estimado, 1),
            "custo_otimizado": round(custo_otimizado, 2),
            "n_geracoes_ga": n_geracoes_total,
            "custo_inicial_ga": round(historico_custos[0], 2) if historico_custos else None,
            "custo_final_ga": round(historico_custos[-1], 2) if historico_custos else None,
        },
        "origem_retorno": hospital_base.nome,
        "sequencia_atendimentos": sequencia_atendimentos,
        "prioridades": {
            "alta": [p.nome for p in rota_otimizada if p.prioridade == 1],
            "media": [p.nome for p in rota_otimizada if p.prioridade == 2],
            "baixa": [p.nome for p in rota_otimizada if p.prioridade == 3],
        },
        "comparacao_baseline_nn": {
            "custo_baseline_nn": round(custo_baseline_nn, 2) if custo_baseline_nn else None,
            "custo_otimizado": round(custo_otimizado, 2),
            "economia_percentual": economia_percentual,
            "ga_superou_nn": (custo_otimizado < custo_baseline_nn) if custo_baseline_nn else None,
        },
        "historico_evolucao": {
            "melhor_custo_por_geracao": [round(c, 2) for c in historico_custos],
            "n_geracoes_total": n_geracoes_total,
            "n_geracoes_com_melhoria": n_geracoes_com_melhoria,
            "n_geracoes_sem_melhoria": n_geracoes_sem_melhoria,
        },
        "vrp": _gerar_secao_vrp(rotas_vrp, resumo_vrp),
    }

    return relatorio



def _gerar_secao_vrp(
    rotas_vrp: Optional[List[List[PontoEntrega]]],
    resumo_vrp: Optional[List[dict]],
) -> Optional[Dict[str, Any]]:
    """Gera a seção VRP do relatório, ou None se não houver dados de VRP."""
    if rotas_vrp is None or resumo_vrp is None:
        return None

    veiculos = []
    for v in resumo_vrp:
        idx = v["veiculo"] - 1
        rota = rotas_vrp[idx]
        veiculos.append({
            "veiculo": v["veiculo"],
            "n_pontos": v["n_pontos"],
            "peso_total": v["peso_total"],
            "capacidade_veiculo": v["capacidade_veiculo"],
            "capacidade_ok": v["capacidade_ok"],
            "distancia_pixels": v["distancia_pixels"],
            "autonomia_veiculo": v["autonomia_veiculo"],
            "autonomia_ok": v["autonomia_ok"],
            "pontos": [
                {
                    "nome": p.nome,
                    "prioridade": p.prioridade,
                    "peso": p.peso,
                    "tempo_atendimento_min": p.tempo_atendimento,
                }
                for p in rota
            ],
        })

    todos_validos = all(v["capacidade_ok"] and v["autonomia_ok"] for v in resumo_vrp)
    return {
        "n_veiculos": len(rotas_vrp),
        "solucao_valida": todos_validos,
        "veiculos": veiculos,
    }
    

def apresentar_relatorio(dados: dict, modo: str = "rich"):
    """
    Apresenta o relatório em diferentes modos:
    - "rich": saída colorida no terminal/notebook
    - "web": saída pensada para Streamlit/Dash
    """
    if modo == "rich":
        # JSON completo
        print_json(data=dados)

        # Seções destacadas
        print("\n[bold cyan]Resumo da Rota:[/bold cyan]")
        print_json(data=dados["resumo"])

        print("\n[bold green]Comparação com Baseline NN:[/bold green]")
        print_json(data=dados["comparacao_baseline_nn"])

        # Tabela resumida
        console = Console()
        table = Table(title="Resumo da Rota")
        table.add_column("Métrica", style="cyan")
        table.add_column("Valor", style="magenta", justify="right")

        resumo = dados["resumo"]
        table.add_row("Total de Pontos", str(resumo["total_pontos_entrega"]))
        table.add_row("Distância (px)", f"{resumo['distancia_total_pixels']:.2f}")
        table.add_row("Tempo Estimado (min)", f"{resumo['tempo_total_estimado_min']:.1f}")
        table.add_row("Custo Otimizado", f"{resumo['custo_otimizado']:.2f}")
        table.add_row("Gerações GA", str(resumo["n_geracoes_ga"]))
        console.print(table)

    elif modo == "web":
        # Aqui você pode usar Streamlit ou Dash
        import streamlit as st
        st.title("Relatório VRP/TSP")
        st.json(dados["resumo"])
        st.json(dados["comparacao_baseline_nn"])
        st.line_chart(dados["historico_evolucao"]["melhor_custo_por_geracao"])

In [ ]:
from groq import Groq
client = Groq(api_key=api_key)
 

dados = gerar_relatorio_rota(
                        rota_otimizada = melhor_rota_refinado,
                        hospital_base = hospital_base_param, 
                        custo_otimizado = melhor_custo_refinado,  
                        historico_custos = histcutomedio1,
                        rota_baseline_nn = rota_nn, 
                        custo_baseline_nn = custo_nn)

# 1. Evolução do custo por geração (linha)
historico = dados["historico_evolucao"]["melhor_custo_por_geracao"]
fig1 = px.line(
    y=historico,
    title="Evolução do Custo por Geração (GA)",
    labels={"y": "Custo"}
)
fig1.show()

# 2. Comparação baseline vs otimizado (barras)
comp = dados["comparacao_baseline_nn"]
fig2 = go.Figure(data=[
    go.Bar(name="Baseline NN", x=["Custo"], y=[comp["custo_baseline_nn"]]),
    go.Bar(name="GA Otimizado", x=["Custo"], y=[comp["custo_otimizado"]])
])
fig2.update_layout(
    barmode="group",
    title="Comparação Baseline NN vs GA Otimizado"
)
fig2.show()

# 3. Sequência de atendimentos (scatter plot)
seq = dados["sequencia_atendimentos"]
fig3 = px.scatter(
    seq,
    x=[p["coords"][0] for p in seq],
    y=[p["coords"][1] for p in seq],
    text=[p["nome"] for p in seq],
    color=[p["prioridade_label"] for p in seq],
    title="Mapa dos Pontos de Atendimento",
    labels={"color": "Prioridade"}
)
fig3.update_traces(textposition="top center")
fig3.show()